# Weak-lensing galaxy shape catalogue validation

In [1]:
# Run the following notebooks in order:
# 1. main_set_up.ipynb (this ncotebook)
# 2. metacal_global.ipynb
# 3. psf_leakage.ipybn
# 4. write_cat.ipynb
# 5. cosmology.ipynb

## Main notebook, set-up, catalogue preparation

### Contents
1. Set-up
2. Load data
3. Matching of stars
4. Select galaxies

In [2]:
import sys
import os
import numpy as np
from astropy.io import fits

In [3]:
from sp_validation.io import *
from sp_validation.cat import *
from sp_validation.survey import *

In [4]:
sp_base= f"{os.environ['HOME']}/astro/repositories/github/sp_validation"

# The following commands will be replaced by import instructions, once the sp_validation scripts are stable
for sc in ['util', 'cat', 'survey', 'galaxy']:
    script = os.path.join(sp_base, 'sp_validation', sc)
    %run $script

## 1. Set-up

In [5]:
# Load parameters
%run params.py

Field name = W3


### Create and open output files and directories

In [6]:
make_out_dirs(plot_dir, plot_subdirs, verbose=verbose)
stats_file = open_stats_file(plot_dir, stats_file_name)

## 2. Load data

### Load merged (final) galaxy catalogue

In [7]:
dd = np.load(galaxy_cat_path, mmap_mode=mmap_mode)

#### Print some quantities to check nothing obvious is wrong with catalogue

In [8]:
print_some_quantities(dd, 'NGMIX_ELL_NOSHEAR', 2, stats_file, invalid=-10, verbose=verbose)

Column names:
('XWIN_WORLD', 'YWIN_WORLD', 'TILE_ID', 'FLAGS', 'IMAFLAGS_ISO', 'NGMIX_MCAL_FLAGS', 'NGMIX_ELL_PSFo_NOSHEAR', 'GALSIM_PSF_ELL_ORIGINAL_PSF', 'SPREAD_CLASS', 'SPREAD_MODEL', 'SPREADERR_MODEL', 'N_EPOCH', 'NGMIX_N_EPOCH', 'NGMIX_ELL_1M', 'NGMIX_ELL_1P', 'NGMIX_ELL_2M', 'NGMIX_ELL_2P', 'NGMIX_ELL_NOSHEAR', 'NGMIX_ELL_ERR_NOSHEAR', 'NGMIX_FLAGS_1M', 'NGMIX_FLAGS_1P', 'NGMIX_FLAGS_2M', 'NGMIX_FLAGS_2P', 'NGMIX_FLAGS_NOSHEAR', 'NGMIX_T_1M', 'NGMIX_T_1P', 'NGMIX_T_2M', 'NGMIX_T_2P', 'NGMIX_T_NOSHEAR', 'NGMIX_T_ERR_1M', 'NGMIX_T_ERR_1P', 'NGMIX_T_ERR_2M', 'NGMIX_T_ERR_2P', 'NGMIX_T_ERR_NOSHEAR', 'NGMIX_Tpsf_1M', 'NGMIX_Tpsf_1P', 'NGMIX_Tpsf_2M', 'NGMIX_Tpsf_2P', 'NGMIX_Tpsf_NOSHEAR', 'NGMIX_FLUX_1M', 'NGMIX_FLUX_1P', 'NGMIX_FLUX_2M', 'NGMIX_FLUX_2P', 'NGMIX_FLUX_NOSHEAR', 'NGMIX_FLUX_ERR_1M', 'NGMIX_FLUX_ERR_1P', 'NGMIX_FLUX_ERR_2M', 'NGMIX_FLUX_ERR_2P', 'NGMIX_FLUX_ERR_NOSHEAR', 'GALSIM_GAL_ELL_1M', 'GALSIM_GAL_ELL_1P', 'GALSIM_GAL_ELL_2M', 'GALSIM_GAL_ELL_2P', 'GALSIM_GAL_ELL_

#### Survey area and potential missing tiles
The approximate observed area is the number of tiles $\times$ 0.25 deg$^2$ (ignoring overlaps and masking).

In [9]:
area_deg2, area_amin2, tile_IDs = get_area(dd, area_tile, verbose=verbose)

Number of tiles found in galaxy catalogue = 214
Area [deg^2] = 53.5


Identify missing tiles by comparing tile ID from catalogue to external input tile ID file.

In [10]:
n_found, n_missing = missing_tiles(tile_IDs, path_tile_ID, path_missing_ID, verbose=verbose)

0/214 = 0% tiles missing


### Load star catalogue

In [11]:
d_star = fits.getdata(star_cat_path, 1)

In [12]:
print_some_quantities(d_star, ['E1_PSF_HSM', 'E2_PSF_HSM'], 1, stats_file, invalid=-10, verbose=verbose)

Column names:
('X', 'Y', 'RA', 'DEC', 'E1_PSF_HSM', 'E2_PSF_HSM', 'SIGMA_PSF_HSM', 'E1_STAR_HSM', 'E2_STAR_HSM', 'SIGMA_STAR_HSM', 'FLAG_PSF_HSM', 'FLAG_STAR_HSM', 'MAG', 'SNR', 'ACCEPTED', 'CCD_NB')

Total number of objects = 90408 = 90 Thousand
Total number of valid objects = 90408 = 90 Thousand
Fraction of invalid objects = 0/90408 = 0%

Mean ellipticity of valid objects:
<e_1> = 0.0243
<e_2> = 0.00937


### 3. Matching of stars

### Matching of star catalogues
Match the star catalogue `d_star` (selected on individual exposures using size-magnitude diagram) to catalogue from tile. Uses some simple criteria to select stars from tile catalogue such as SPREAD_CLASS.

This is mainly for testing, this match will not be used later.

#### Match to all objects

In [13]:
ind_star, mask_area_tiles, n_star_tot = check_matching(d_star, dd, ['RA', 'DEC'], ['XWIN_WORLD', 'YWIN_WORLD'], thresh,
                                                       stats_file, name=None, verbose=verbose)

Number of matched stars from exposures to total catalogue = 64997/90408 = 71.9%
Number of matched stars after removing multiple matches = 54486/90408 = 60.3%


#### Refine: Match to valid, unflagged ngmix sample

In [14]:
# Flags to indicate valid ngmix star sample
m_star_ngmix = (dd['FLAGS'][ind_star] == 0) & \
               (dd['IMAFLAGS_ISO'][ind_star] == 0) & \
               (dd['NGMIX_MCAL_FLAGS'][ind_star] == 0) & \
               (dd['NGMIX_ELL_PSFo_NOSHEAR'][:,0][ind_star] != -10)

print_stats('ngmix:', stats_file, verbose=verbose)

ra_star_ngmix, dec_star_ngmix, g_star_psf_ngmix = \
    match_subsample(dd, ind_star, m_star_ngmix, ['XWIN_WORLD', 'YWIN_WORLD'], 'NGMIX_ELL_PSFo_NOSHEAR',
                    n_star_tot, stats_file, verbose=verbose)

ngmix:
Number of stars matched to valid sample = 50293/90408 = 55.6%


#### Refine: Match to valid, unflagged galsim sample

In [15]:
m_star_galsim = (dd['FLAGS'][ind_star] == 0) & \
                (dd['IMAFLAGS_ISO'][ind_star] == 0) & \
                (dd['GALSIM_PSF_ELL_ORIGINAL_PSF'][:,0][ind_star] != -10)

print_stats('galsim:', stats_file, verbose=verbose)

ra_star_galsim, dec_star_galsim, g_star_psf_galsim = \
    match_subsample(dd, ind_star, m_star_galsim, ['XWIN_WORLD', 'YWIN_WORLD'], 'GALSIM_PSF_ELL_ORIGINAL_PSF',
                    n_star_tot, stats_file, verbose=verbose)

galsim:
Number of stars matched to valid sample = 46486/90408 = 51.4%


#### Refine: Match to ngmix SPREAD_CLASS samples

In [16]:
print_stats('ngmix:', stats_file, verbose=verbose)
match_spread_class(dd, ind_star, m_star_ngmix, stats_file, len(ra_star_ngmix), verbose=verbose)

ngmix:
Number of stars selected as star (SPREAD_CLASS=0)   = 49999/50293 = 99.4%
Number of stars selected as galaxy (SPREAD_CLASS=1) = 0/50293 = 0.0%
Number of stars selected as other (SPREAD_CLASS=2)  = 294/50293 = 0.6%


#### Refine: Match to galsim SPREAD_CLASS samples

In [17]:
print_stats('galsim:', stats_file, verbose=verbose)
match_spread_class(dd, ind_star, m_star_galsim, stats_file, len(ra_star_galsim), verbose=verbose)

galsim:
Number of stars selected as star (SPREAD_CLASS=0)   = 46343/46486 = 99.7%
Number of stars selected as galaxy (SPREAD_CLASS=1) = 0/46486 = 0.0%
Number of stars selected as other (SPREAD_CLASS=2)  = 143/46486 = 0.3%


## X. Check for objects with invalid PSF

In [18]:
check_invalid(dd, ['NGMIX_ELL_PSFo_NOSHEAR', 'NGMIX_ELL_NOSHEAR'], [0, 0], [-10, -10],
              stats_file, name=['PSF', 'galaxy ellipticity'], verbose=verbose)

Invalid PSF found for 158639/5367032 = 0.03% objects
Invalid galaxy ellipticity found for 158639/5367032 = 0.03% objects


## 4. Select galaxies

### 4.1 Using the spread model parameter
This parameter quantifies the size of an object with respect to the local PSF. Objects with larger spread model are more likely to be galaxies.

#### Common flags and cuts
First, set cuts common to ngmix and galsim:
  - spread model: select objects well larger than the PSF
  - magnitude: cut galaxies that are too faint (= too noisy, likely to be
    artefacts), and too bright (might be too large for postage stamp)
  - flags: cut objects that were flagged as invalid or masked
  - n_epoch: select objects observed on at leatst one epoch (for safety,
    to avoid potential errors with empty data)

In [19]:
cut_common = classification_galaxy_base(dd)

#### ngmix

In [20]:
# add ngmix-specific cuts: select objects with valid flags, valid PSF ellipticity, valid moments,
# ngmix used at least one epoch

m_gal_ngmix = classification_galaxy_ngmix(dd, cut_common, stats_file, verbose=verbose)

ngmix: Objects selected as galaxies = 3466497/5367032 = 64.6%


#### galsim

In [21]:
# add ngmix-specific cuts: select objects with valid PSF ellipticity

m_gal_galsim = classification_galaxy_galsim(dd, cut_common, stats_file, verbose=verbose)

galsim: Objects selected as galaxies = 3330199/5367032 = 62.0%
